In [3]:
!pip install -q pyspark

import os
import shutil
import glob
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

os.makedirs("data_lake/bronze", exist_ok=True)
os.makedirs("data_lake/silver/preprocessed_sales", exist_ok=True)
os.makedirs("silver", exist_ok=True)

csv_files = [f for f in os.listdir(".") if f.startswith("sales") and f.endswith(".csv")]
bronze_path = "data_lake/bronze/sales.csv"
shutil.copy(csv_files[0], bronze_path)

spark = SparkSession.builder.appName("Data_Mining_Spark_Data_Lake").getOrCreate()
spark.conf.set("spark.sql.ansi.enabled", "false")

df_bronze = spark.read.csv(bronze_path, header=True, inferSchema=True)

df_bronze.printSchema()
df_bronze.show(5, truncate=False)

total_bronze_count = df_bronze.count()
print("Total Record Count (Bronze):", total_bronze_count)

distinct_count = df_bronze.distinct().count()
print("Exact Duplicate Rows:", total_bronze_count - distinct_count)

null_counts = df_bronze.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_bronze.columns])
null_counts.show()

df_bronze.groupBy("category").count().orderBy("category").show(20, truncate=False)
df_bronze.groupBy("city").count().orderBy("city").show(20, truncate=False)
df_bronze.filter(F.col("customer_name") != F.trim(F.col("customer_name"))).select("order_id", "customer_name").show(truncate=False)

print("Invalid Quantity (<= 0):", df_bronze.filter(F.col("quantity") <= 0).count())
df_bronze.filter(F.col("quantity") <= 0).select("order_id", "product", "quantity", "unit_price").show(5)

print("Invalid Unit Price (<= 0):", df_bronze.filter(F.col("unit_price") <= 0).count())
df_bronze.filter(F.col("unit_price") <= 0).select("order_id", "product", "quantity", "unit_price").show(5)

print("Invalid Discount (< 0 or > 100):", df_bronze.filter((F.col("discount_percent") < 0) | (F.col("discount_percent") > 100)).count())
df_bronze.filter((F.col("discount_percent") < 0) | (F.col("discount_percent") > 100)).select("order_id", "product", "discount_percent").show(10)

valid_date_regex = r"^(202[0-9]-(0[13578]|1[02])-(0[1-9]|[12][0-9]|3[01])|202[0-9]-(0[469]|11)-(0[1-9]|[12][0-9]|30)|202[0-9]-02-(0[1-9]|1[0-9]|2[0-8]))$"
df_bronze.filter(~F.col("order_date").rlike(valid_date_regex)).select("order_id", "order_date").show(truncate=False)

df_silver = df_bronze.dropDuplicates()

text_cols = ["customer_id", "customer_name", "product", "category", "payment_method", "city", "state", "order_status"]
for c in text_cols:
    df_silver = df_silver.withColumn(c, F.regexp_replace(F.trim(F.col(c)), r"\s+", " "))

for c in ["customer_name", "category", "city", "state", "order_status"]:
    df_silver = df_silver.withColumn(c, F.initcap(F.col(c)))

df_silver = df_silver.withColumn(
    "payment_method",
    F.when(F.upper(F.col("payment_method")) == "UPI", F.lit("UPI"))
     .when(F.lower(F.col("payment_method")) == "cash on delivery", F.lit("Cash on Delivery"))
     .otherwise(F.initcap(F.col("payment_method")))
)

df_silver = df_silver.fillna({
    "customer_name": "Unknown",
    "payment_method": "Unknown",
    "city": "Unknown"
})

df_silver = df_silver.withColumn("quantity", F.abs(F.col("quantity")))
df_silver = df_silver.withColumn("unit_price", F.abs(F.col("unit_price")))
df_silver = df_silver.filter((F.col("quantity") > 0) & (F.col("unit_price") > 0))

df_silver = df_silver.withColumn(
    "discount_percent",
    F.when(F.col("discount_percent") < 0, 0)
     .when(F.col("discount_percent") > 100, 100)
     .otherwise(F.col("discount_percent"))
)

df_silver = df_silver.withColumn(
    "order_date",
    F.when(F.col("order_date") == "2025/07/14", F.lit("2025-07-14"))
     .when(F.col("order_date") == "09-15-2026", F.lit("2026-09-15"))
     .when(F.col("order_date").rlike(valid_date_regex), F.col("order_date"))
     .otherwise(F.lit(None))
)

df_silver = df_silver.filter(F.col("order_date").isNotNull())

silver_dir = "data_lake/silver/preprocessed_sales"
df_silver.coalesce(1).write.mode("overwrite").option("header", "true").csv(silver_dir)

part_file = glob.glob(f"{silver_dir}/part-*.csv")[0]
shutil.copy(part_file, "silver/sales_silver.csv")

silver_count = df_silver.count()
print("Row Count Before Preprocessing (Bronze):", total_bronze_count)
print("Row Count After Preprocessing (Silver):", silver_count)

df_silver_verify = spark.read.csv(silver_dir, header=True, inferSchema=True)
df_silver_verify.show(10, truncate=False)

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------

### Summary of Preprocessing Operations Performed
1. Removed 10 exact duplicate rows using `dropDuplicates()`.
2. Trimmed extra spaces and standardized capitalization across text fields (`customer_name`, `category`, `city`, `state`, `order_status`, and `payment_method`).
3. Handled missing values in `customer_name` (10), `payment_method` (8), and `city` (8) by imputing `"Unknown"`.
4. Converted negative `quantity` and `unit_price` entries to positive values (`abs()`), filtered out 10 rows with `0` quantity or `0` unit price, and clamped `discount_percent` to `[0, 100]`.
5. Standardized valid mixed-format dates (`2025/07/14`, `09-15-2026`) to `yyyy-MM-dd` and filtered out 4 invalid dates (`31/02/2026`, `not-a-date`, `2025-02-30`, `2026-13-05`), resulting in 976 Silver records.